# EDA — Palmer Penguins

**Objetivo:** responder seis preguntas sobre el dataset y terminar en una
tabla de decisiones que justifique `configs/base.toml` y `transforms.py`.

Este notebook es exploración, no código de producción. No se importa desde `src/`.

## 1. Carga y forma

**Pregunta:** ¿qué tengo entre manos?

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

# Funciona tanto si Jupyter se lanza desde la raíz del repo
# como si se lanza desde notebooks/.
REPO_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DATA_PATH = REPO_ROOT / "data" / "raw" / "penguins.csv"

df = pd.read_csv(DATA_PATH)

print(f"Archivo: {DATA_PATH}")
print(f"Forma:   {df.shape}")
df.dtypes

## 2. Valores faltantes

**Pregunta:** ¿hace falta imputar o puedo eliminar filas?

**Cómo leerlo:** si los nulos son pocos y están dispersos, imputar. Si una columna
está vacía en más del ~50%, probablemente sobra.

In [ ]:
df.isna().sum().sort_values(ascending=False)

In [ ]:
nulos = df.isna().sum().sort_values(ascending=False)
nulos = nulos[nulos > 0]

if nulos.empty:
    print("No hay valores faltantes")
else:
    fig, ax = plt.subplots(figsize=(7, 3))
    barras = ax.barh(nulos.index, nulos.values, color="#4C72B0")
    ax.bar_label(barras)
    ax.invert_yaxis()
    ax.set_xlabel("Valores faltantes")
    ax.set_title("Valores faltantes por columna")
    ax.grid(axis="x", alpha=0.3)
    plt.tight_layout()
    plt.show()

## 3. Cardinalidad de categóricas

**Pregunta:** ¿cuántas categorías hay por columna?

**Cómo leerlo:** pocas categorías (2–10) → one-hot. Muchas (cientos) → target encoding.

In [ ]:
categoricas = ["island", "sex"]

for columna in categoricas:
    print(columna, "->", df[columna].nunique(), "categorías")
    print(df[columna].value_counts(dropna=False))
    print()

In [ ]:
fig, axes = plt.subplots(1, len(categoricas), figsize=(9, 3))

for ax, columna in zip(axes, categoricas):
    conteo = df[columna].value_counts(dropna=False)
    etiquetas = [str(valor) for valor in conteo.index]
    barras = ax.bar(etiquetas, conteo.values, color="#55A868")
    ax.bar_label(barras)
    ax.set_title(columna)
    ax.set_ylabel("filas")
    ax.tick_params(axis="x", rotation=30)

plt.tight_layout()
plt.show()

## 4. Balance de clases

**Pregunta:** ¿alguna clase domina?

**Cómo leerlo:** si una clase supera el ~60%, una partición aleatoria puede dejar
folds sin esa clase. La estratificación lo evita.

In [ ]:
df["species"].value_counts()
df["species"].value_counts(normalize=True).round(3)

In [ ]:
conteo = df["species"].value_counts()

fig, ax = plt.subplots(figsize=(6, 3))
barras = ax.bar(conteo.index, conteo.values, color="#C44E52")
ax.bar_label(barras)
ax.set_ylabel("filas")
ax.set_title("Balance de clases")
plt.tight_layout()
plt.show()

## 5. Distribuciones y outliers

**Pregunta:** ¿hay valores extremos? ¿Las escalas son comparables?

**Cómo leer una caja y bigotes:**

- La **caja** va del primer al tercer cuartil (Q1–Q3): contiene el 50% central.
- La **línea dentro** de la caja es la mediana.
- Los **bigotes** llegan hasta 1.5 × IQR desde la caja.
- Los **puntos** fuera de los bigotes son outliers.

Si los bigotes de una columna son asimétricos o hay muchos puntos sueltos, la mediana
es mejor imputador que la media.

In [ ]:
numericas = ["bill_length_mm", "bill_depth_mm", "flipper_length_mm", "body_mass_g"]

df[numericas].describe().round(2)

In [ ]:
especies = sorted(df["species"].dropna().unique())

fig, axes = plt.subplots(1, 4, figsize=(15, 4))

for ax, columna in zip(axes, numericas):
    datos = [df.loc[df["species"] == especie, columna].dropna().values for especie in especies]
    ax.boxplot(datos, patch_artist=True)
    ax.set_title(columna)
    ax.set_xticks(range(1, len(especies) + 1))
    ax.set_xticklabels(especies, rotation=30)
    ax.grid(axis="y", alpha=0.3)

fig.suptitle("Caja y bigotes por especie")
plt.tight_layout()
plt.show()

## 6. Columnas inútiles

**Pregunta:** ¿alguna columna aporta algo?

**Importante:** `year` describe *cuándo se recolectó*, no al pingüino. Incluirlo permite
que el modelo aprenda artefactos del muestreo, no biología.

In [ ]:
print("años distintos:", df["year"].nunique())
df.groupby("year")["species"].value_counts()

## 7. Tabla de decisiones

**Números reales** verificados sobre `data/raw/penguins.csv`:

| Observación | Decisión | Dónde se aplica |
|---|---|---|
| 11 nulos en `sex`; 2 en cada medida | Imputar, no eliminar filas | `SimpleImputer` |
| `sex` es la única categórica con nulos | Imputar con la moda | `strategy="most_frequent"` |
| `island` tiene 3 valores y `sex` 2 | One-hot tolerante a categorías nuevas | `OneHotEncoder(handle_unknown="ignore")` |
| Clases 44% / 36% / 20% (Adelie / Gentoo / Chinstrap) | CV estratificada | `folds` + `stratify` |
| `body_mass_g`: 2700–6300, media 4202 ≠ mediana 4050 | Imputar con mediana | `strategy="median"` |
| Escalas de 13 a 6300 (≈300×) | Escalar | `StandardScaler` |
| `year` solo tiene 3 valores y describe el muestreo | Excluir de features | `features` |
| 8 columnas: 6 features, 1 target, 1 excluida | Declarar features explícitamente | `features` / `target` |

**Regla:** cada fila debe poder explicarse señalando una celda de arriba.
Si no puedes mostrar la evidencia, la decisión sobra.